In [ ]:
!pip install timm albumentations flask

In [ ]:
import os
from glob import glob
import json

model_dir = '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/train_results'
annotations_path = os.path.join(model_dir, 'annotations.json')
with open(annotations_path) as f:
    annotations = json.load(f)

print(f"이미지 개수: {len(annotations)}")

In [ ]:
annotations[:2]

In [ ]:
!FLASK_ENV=development FLASK_APP=/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/part3_chapter03_app/app.py flask run

In [ ]:
import subprocess

flask_command = ["python3", "/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/part3_chapter03_app/app.py"]

flask_process = subprocess.Popen(flask_command)

In [ ]:
import requests
import io
from PIL import Image
import random
from tqdm import tqdm
import numpy as np

feature_map = {}
margin = 0.1
random.shuffle(annotations)

for annot in tqdm(annotations[:10]):
    image_path = annot['image_path']
    image_id = annot['image_id']
    image = Image.open(image_path).convert('RGB')
    box = annot['box']

    w = box[2] - box[0]
    h = box[3] - box[1]

    new_box = [
        int(box[0] - w * margin),
        int(box[1] - h * margin),
        int(box[2] + w * margin),
        int(box[3] + h * margin),
    ]
    cropped_image = image.crop(new_box)

    image_bytes = io.BytesIO()
    cropped_image.save(image_bytes, format='JPEG')

    files = {'file': ('test_image.jpg', image_bytes.getvalue())}

    resp = requests.post(url="http://localhost:5000/predict", files=files)

    result = resp.json()
    feature_map[f"{image_path}_{image_id}"] = result["feature"]


In [ ]:
flask_process.terminate()

In [ ]:
feature_map.keys()

In [ ]:
import numpy as np

feat = np.array(feature_map[''])
feat.shape

In [ ]:
print(feat)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
import numpy as np

class BatchJsonDataset(Dataset):
    def __init__(self, annotations margin=0.1, transform=None):
        self.annotations = annotations
        self.margin = margin
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        annot = self.annotations[index]
        image_path = os.path.join(annot['image_path'])
        image_id = annot['image_id']
        image = Image.open(image_path).convert("RGB")
        box = annot['box']

        w = box[2] - box[0]
        h = box[3] - box[1]

        new_box = [int(box[0] - w * self.margin),
                   int(box[1] - h * self.margin),
                   int(box[2] + w * self.margin),
                   int(box[3] + h * self.margin)]

        cropped_image = image.crop(new_box)

        if self.transform:
            try:
                cropped_image = self.transform(image=np.array(cropped_image))['image']
                image = self.transform(image=np.array(image))['image']
            except:
                black_image = Image.new('RGB', (224, 224), (0, 0, 0))
                cropped_image = self.transfrom(image=np.array(black_image))['image']
                print(idx, annot)


        return cropped_image, image, image_path_id

In [ ]:
import torch.nn as nn

class BranchClassifier(nn.Module):
    def __init__(self, num_classes_list, num_branches=4, model_name='efficientnet_b0', pretrained=True):
        super(BranchClassifier, self).__init__()

        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1280, 256),
                nn.SiLU(),
                nn.Dropout(0.3),
                nn.Linear(256, num_classes_list[i])
            )
            for i in range(num_branches)
        ])

    def forward(self, x):
        features = self.backbone(x)
        outputs = [branch(features) for branch in self.branches]

        return outputs

In [ ]:
import os
import torch
import json
import albumentations as A
from albumentations.pytorch import ToTensorV2

model_dir = '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/train_results'
device = torch.device('cuda' if torch.cuda.is_available() else 'gpu')

num_branches = 4

with open(os.path.join(model_dir, 'detail_category_list.json'), 'r') as json_f:
    detail_category_list = json.load(json_f)
with open(os.path.join(model_dir, 'color_list.json'), 'r') as json_f:
    color_list = json.load(json_f)
with open(os.path.join(model_dir, 'fit_list.json'), 'r') as json_f:
    fit_list = json.load(json_f)
with open(os.path.join(model_dir, 'length_list.json'), 'r') as json_f:
    length_list = json.load(json_f)

num_classes_list = [len(detail_category_list), len(color_list), len(fit_list), len(length_list)]
model = BranchClassifier(num_classes_list = num_classes_list).eval().to(device)
weight = torch.load(os.path.join(model_dir, 'best_model.pth'), map_location='cpu')
model.load_state_dict(weight)

transform = A.Compose([
    A.LongestMaxSize(max_size=224, always_apply=True),
    A.PadIfNeeded(min_height=224, min_width=224, always_apply=True, border_mode=0),
    A.Normalize(p=0.1, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])


In [ ]:
dataset = BatchJsonDataset(annotations, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=36, num_workers=4, shuffle=True)

In [ ]:
feature_map = {}
model.eval()

with torch.no_grad():
    for idx, (cropped_images, imaeges, image_path_id) in tqdm(enumerate(dataloader)):
        cropped_images = cropped_images.to(device)
        outputs, features = model(cropped_images)
        for batch_idx in range(len(features)):
            feature_map[image_path_id[batch_idx]] = features[batch_idx].cpu().numpy().tolist()

In [ ]:
list(feature_map.keys())[:2]

In [ ]:
vector_dir = '/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/vectors'
os.makedirs (vector_dir, exist_ok=True)
with open('/content/drive/MyDrive/Colab Notebooks/fast_campus_image_processing/similar_fashion_products/vectors', 'w') as f:
    json.dump(feature_map, f, indent='\t', ensure_ascii=False)